# Updated Hierarchical Model (with feature engineering from `ass1.ipynb`)

This notebook ports the logic of `Hier_Model.R` to Python and folds in the feature-engineering
steps developed in `ass1.ipynb`, so the academic / family / lifestyle sub-models are trained on
engineered rather than raw features.

| Feature | Applied to |
|---|---|
| `log1p(absences)` + `no_absences` flag | **Academic** block (replaces raw `absences`) |
| `had_failure` = `failures > 0` | **Academic** block (kept alongside raw `failures` as a threshold effect) |
| `Medu × Fedu` interaction | **Family** block (added alongside `Medu`, `Fedu` individually) |
| One-hot encoding, fit fresh inside every CV fold (no leakage) | All three blocks + the flat baseline |

**Two assumptions worth checking against your R environment**, since they weren't defined in the
`Hier_Model.R` snippet you sent:
- `guardian_edu` and `age_std` aren't columns in the raw `train.rds`/`test.rds` used by `ass1.ipynb`.
  This notebook uses `Medu` + `Fedu` (+ their interaction) directly in the family block instead of
  `guardian_edu`, and standardises `age` inside each fold's pipeline instead of a precomputed `age_std`.
  If your `guardian_edu` was built differently, swap it back in.
- `baseline_lm_cv` (fit via `caret` in your R script) wasn't included in the file you sent, so this
  notebook refits an equivalent **flat OLS baseline** — same engineered features, same 10 folds — so
  the MSE comparison is self-contained and reproducible without needing that object.

In [ ]:
import numpy as np
import pandas as pd
import pyreadr

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold

SEED = 123      # matches set.seed(123) in Hier_Model.R
N_FOLDS = 10    # matches createFolds(..., k = 10)

train = pyreadr.read_r("train.rds")[None]
test  = pyreadr.read_r("test.rds")[None]

X_raw = train.drop(columns="score")
y = train["score"].to_numpy()
print(train.shape, test.shape, "missing values:", int(train.isna().sum().sum()), int(test.isna().sum().sum()))
train.head()

## 1. Feature engineering

Same `engineer()` used in `ass1.ipynb`, reused unchanged so the two notebooks stay consistent.

In [ ]:
def engineer(df):
    """Feature engineering from ass1.ipynb, applied once to the whole predictor frame."""
    d = df.copy()
    d["log_absences"] = np.log1p(d["absences"])
    d["no_absences"]  = (d["absences"] == 0).astype(float)
    d["had_failure"]  = (d["failures"] > 0).astype(float)
    d["Medu_x_Fedu"]  = d["Medu"] * d["Fedu"]
    return d.drop(columns="absences")

train_eng = engineer(X_raw)
train_eng["score"] = y
test_eng = engineer(test)
train_eng.head()

## 2. Hierarchical sub-model blocks

Each block mirrors the corresponding `lm(...)` formula in `Hier_Model.R`, updated to draw on the
engineered columns instead of the raw ones:

| Block | Original R formula | Updated columns |
|---|---|---|
| **Academic** | `school + reason + studytime + failures + schoolsup + paid + higher + absences + traveltime` | swaps `absences` → `log_absences` + `no_absences`, adds `had_failure` |
| **Family** | `guardian_edu + guardian + Mjob + Fjob + famsize + Pstatus + famrel + nursery` | swaps `guardian_edu` → `Medu` + `Fedu` + `Medu_x_Fedu` |
| **Lifestyle** | `age_std + sex + address + health + romantic + activities + internet + freetime + goout + Dalc + Walc` | unchanged; `age` is standardised inside the pipeline instead of a precomputed `age_std` |

In [ ]:
ACADEMIC_COLS = ["school", "reason", "studytime", "failures", "had_failure",
                 "schoolsup", "paid", "higher", "log_absences", "no_absences", "traveltime"]
FAMILY_COLS = ["Medu", "Fedu", "Medu_x_Fedu", "guardian", "Mjob", "Fjob",
               "famsize", "Pstatus", "famrel", "nursery"]
LIFESTYLE_COLS = ["age", "sex", "address", "health", "romantic", "activities",
                  "internet", "freetime", "goout", "Dalc", "Walc"]

BLOCKS = {"academic": ACADEMIC_COLS, "family": FAMILY_COLS, "lifestyle": LIFESTYLE_COLS}
ALL_COLS = sorted(set(ACADEMIC_COLS) | set(FAMILY_COLS) | set(LIFESTYLE_COLS))


def make_sub_pipeline(cols, df_for_dtypes):
    """One-hot encode the categorical columns in `cols`, passthrough the numeric ones,
    then plain OLS -- refit fresh inside every fold, exactly like the R script's
    cv_academic / cv_family / cv_lifestyle lm() calls."""
    cat_cols = [c for c in cols if str(df_for_dtypes[c].dtype) in ("category", "object")]
    ct = ColumnTransformer(
        [("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False), cat_cols)],
        remainder="passthrough")
    return Pipeline([("ct", ct), ("lm", LinearRegression())])

## 3. 10-fold cross-validation: hierarchical model

Direct translation of the `for (i in seq_along(folds))` loop in `Hier_Model.R`: sub-models are
refit on the CV-training fold only, their predictions become "proxies", and a master OLS model
combines the three proxies. Nothing is fit on data outside the current fold.

In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

hier_fold_mse = []

for train_idx, test_idx in kf.split(train_eng):
    cv_train = train_eng.iloc[train_idx]
    cv_test  = train_eng.iloc[test_idx]

    proxies_train, proxies_test = {}, {}
    for block_name, cols in BLOCKS.items():
        sub_pipe = make_sub_pipeline(cols, cv_train)
        sub_pipe.fit(cv_train[cols], cv_train["score"])
        proxies_train[f"{block_name}_proxy"] = sub_pipe.predict(cv_train[cols])
        proxies_test[f"{block_name}_proxy"]  = sub_pipe.predict(cv_test[cols])

    master_train = pd.DataFrame(proxies_train, index=cv_train.index)
    master_test  = pd.DataFrame(proxies_test, index=cv_test.index)

    master = LinearRegression().fit(master_train, cv_train["score"])
    preds = master.predict(master_test)

    fold_mse = float(np.mean((preds - cv_test["score"].to_numpy()) ** 2))
    hier_fold_mse.append(fold_mse)

hierarchical_mse = float(np.mean(hier_fold_mse))
print(f"Hierarchical model MSE (10-fold CV): {hierarchical_mse:.4f}")

## 4. 10-fold cross-validation: baselines

Two reference points, computed on the **same folds** as the hierarchical model so the comparison is
paired and fair:
- **Mean baseline** -- predicting the training-fold mean score (matches the "predict the mean" baseline in `ass1.ipynb`).
- **Flat OLS baseline** -- a single OLS model on all engineered predictors combined, standing in for the
  `baseline_lm_cv` object from your R script (not included in the file you sent).

In [ ]:
baseline_fold_mse, mean_fold_mse = [], []

for train_idx, test_idx in kf.split(train_eng):
    cv_train = train_eng.iloc[train_idx]
    cv_test  = train_eng.iloc[test_idx]

    base_pipe = make_sub_pipeline(ALL_COLS, cv_train)
    base_pipe.fit(cv_train[ALL_COLS], cv_train["score"])
    base_preds = base_pipe.predict(cv_test[ALL_COLS])
    baseline_fold_mse.append(float(np.mean((base_preds - cv_test["score"].to_numpy()) ** 2)))

    mean_pred = cv_train["score"].mean()
    mean_fold_mse.append(float(np.mean((mean_pred - cv_test["score"].to_numpy()) ** 2)))

baseline_mse = float(np.mean(baseline_fold_mse))
mean_baseline_mse = float(np.mean(mean_fold_mse))

## 5. Final comparison

Same layout as the `cat(...)` block at the end of `Hier_Model.R`.

In [ ]:
print("=== 10-Fold Cross-Validation MSE Comparison ===")
print("Mean baseline MSE:      ", round(mean_baseline_mse, 4))
print("Flat OLS baseline MSE:  ", round(baseline_mse, 4))
print("Hierarchical model MSE: ", round(hierarchical_mse, 4))

## 6. Optional: refit on all training data and predict `test.rds`

Not part of `Hier_Model.R`, but included in case you also need test-set predictions in the same
format as `ass1.ipynb` (`predictions.rds`, a plain numeric vector of length 79). Delete this section
if you only needed the CV comparison above.

In [ ]:
final_proxies_train, final_proxies_test = {}, {}
for block_name, cols in BLOCKS.items():
    sub_pipe = make_sub_pipeline(cols, train_eng)
    sub_pipe.fit(train_eng[cols], train_eng["score"])
    final_proxies_train[f"{block_name}_proxy"] = sub_pipe.predict(train_eng[cols])
    final_proxies_test[f"{block_name}_proxy"]  = sub_pipe.predict(test_eng[cols])

final_master_train = pd.DataFrame(final_proxies_train, index=train_eng.index)
final_master_test  = pd.DataFrame(final_proxies_test, index=test_eng.index)

final_master = LinearRegression().fit(final_master_train, train_eng["score"])
pred = final_master.predict(final_master_test)
assert pred.shape == (len(test_eng),) and np.isfinite(pred).all()
print("Predictions:\n", pred)

In [ ]:
import subprocess, shutil, glob, os

pd.Series(pred).to_csv("_pred_tmp.csv", index=False, header=False, float_format="%.10f")
rscript = shutil.which("Rscript") or (sorted(glob.glob(r"C:\\Program Files\\R\\R-*\\bin\\Rscript.exe")) or [None])[-1]
print("Rscript:", rscript)
subprocess.run([rscript, "-e",
                "x <- scan(\'_pred_tmp.csv\', quiet=TRUE); saveRDS(as.numeric(x), \'predictions.rds\'); "
                "y <- readRDS(\'predictions.rds\'); cat(\'class:\', class(y), \' length:\', length(y), \'\\n\'); print(summary(y))"],
               check=True)
os.remove("_pred_tmp.csv")